# Diffusion Long-Form Compare on Downloads Songs

This notebook runs the same random long-form jobs from `Downloads/` through multiple diffusion checkpoints.

Default panel:
- `best`
- `epoch_005`

The default config below is intentionally **stability-biased**:
- shorter excerpts
- lower `t_start`
- more frequent re-anchoring
- stronger source blending
- `wave` assembly to avoid the heavy full-track mel vocode path

Outputs:
- per-job long-form folders from the Lab 4 coherence runner
- `manifest.csv` with source, target, checkpoint label, and output paths
- `summary.json` and `checkpoint_panel.json`


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 4').exists() and (path / 'lab 3.1').exists():
            return path
    raise RuntimeError('Could not resolve repo root from current working directory.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import diffusion_longform_compare as dlc
importlib.reload(dlc)
REPO

In [ ]:
cfg = dlc.DiffusionLongformCompareConfig(
    downloads_dir=Path.home() / 'Downloads',
    run_dir=None,        # None = auto-pick most recent diffusion run
    cache_dir=None,
    n_songs=2,
    targets_per_song=2,
    source_seconds=45.0,
    chunk_seconds=3.0,
    overlap_seconds=0.5,
    t_start=240,
    t_start_end=180,
    reanchor_every=4,
    reanchor_t_start=160,
    ddim_steps=50,
    guidance_scale=1.75,
    style_strength=0.60,
    source_prefix_blend=0.45,
    source_mel_blend=0.10,
    hf_source_blend=0.18,
    hf_start_bin=56,
    mel_time_smooth=3,
    mel_freq_smooth=0,
    assemble_domain='mel',
    device='auto',
    seed=328,
)
RUN_ALL = True

ctx = dlc.ddb.resolve_inference_context(
    dlc.ddb.DiffusionDownloadsBatchConfig(run_dir=cfg.run_dir, cache_dir=cfg.cache_dir)
)
print('Resolved run dir:    ', ctx['run_dir'])
print('Resolved cache dir:  ', ctx['cache_dir'])
print('Planned output root: ', cfg.output_root / cfg.tag)

In [ ]:
checkpoint_panel = dlc.resolve_checkpoint_panel(cfg, include_best=True, include_epoch5=True)
pd.DataFrame([
    {'label': row['label'], 'path': str(row['path'])}
    for row in checkpoint_panel
])

In [ ]:
jobs = dlc.plan_longform_jobs(cfg)
display(pd.DataFrame(jobs))
print('Total planned long-form jobs:', len(jobs))
print('Total runs across checkpoints:', len(jobs) * len(checkpoint_panel))

In [ ]:
summary = None
if RUN_ALL:
    summary = dlc.run_longform_compare(cfg, checkpoint_panel)
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to launch the long-form compare batch.')

In [ ]:
summary_path = cfg.output_root / cfg.tag / 'summary.json'
manifest_path = cfg.output_root / cfg.tag / 'manifest.csv'
if summary_path.exists():
    print(summary_path)
    print(manifest_path)
    display(pd.read_csv(manifest_path))
else:
    print('No outputs yet for this tag.')